In [1]:
import pathlib
import sys

PACKAGE_PATH = pathlib.Path.cwd().parent.resolve()
assert PACKAGE_PATH.exists()

sys.path.append(str(PACKAGE_PATH))

In [2]:
import json

In [3]:
from CogniScan.encoder import Encoder
from CogniScan.index import RamIndex
from CogniScan.wrappers.cogniscan_onto_wrapper import CogniScanOntoWrapper

/home/smertlove/sandbox/hse/anthologies/CogniScan/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
DATADIR = pathlib.Path.cwd().parent.resolve() / "data"
assert DATADIR.exists()

### Загрузка онтологии

In [5]:
onto = CogniScanOntoWrapper(str(DATADIR / "merged_ontology.rdf"))

Failed to convert Literal lexical form to value. Datatype=http://www.w3.org/2001/XMLSchema#decimal, Converter=<class 'decimal.Decimal'>
Traceback (most recent call last):
  File "/home/smertlove/sandbox/hse/anthologies/CogniScan/venv/lib/python3.10/site-packages/rdflib/term.py", line 2262, in _castLexicalToPython
    return conv_func(lexical)  # type: ignore[arg-type]
decimal.InvalidOperation: [<class 'decimal.ConversionSyntax'>]


       ONTOLOGY STATISTICS       
Classes              :     58
Object Properties    :      7
Datatype Properties  :      7
Individuals          :    434
Total Statements     :   2118


### Загрузка кодировщика

In [6]:
encoder = Encoder("alexyalunin/RuBioRoBERTa")

Some weights of RobertaModel were not initialized from the model checkpoint at alexyalunin/RuBioRoBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Построение поискового индекса

In [7]:
demo_data_path = DATADIR / "abstracts_demo_1400.json"

with open(demo_data_path, "r", encoding="utf-8") as file:
    demo_data = json.load(file)

len(demo_data)

1428

In [8]:
texts = [
    entry["abstract"]
    for entry
    in demo_data
]

meta = [None] * len(texts)

In [9]:
index = RamIndex()

In [10]:
index.build_index(
    texts,
    meta,
    encoder,
)

### Интерфейс к онтологии

In [11]:
def denormalize(s: str):
    return s.lower().replace("_", " ")

In [12]:
def get_diseases_data_from_onto(
    query: str,
    onto: CogniScanOntoWrapper,
    delimiter = ",",
    method="shallow",
):

    if method not in ("shallow", "shallow_described", "deep"):
        raise ValueError(f"No such method: {repr(method)}")

    symptoms = query.split(delimiter)

    diseases = []
    for symptom in symptoms:
        if symptom.strip():
            diseases.extend(
                onto.get_diseases_by_symptom(
                    symptom.strip()
                )
            )

    if method == "shallow":
        return list(map(denormalize, diseases))

    descriptions = []
    for disease in diseases:
        descriptions.append(onto.get_description_of_disease(disease)[0])

    symptoms_join = ", ".join(symptoms)

    if method == "shallow_described":
        return [symptoms_join] + descriptions

    parents = []
    for disease in diseases:
        for parent, comment in onto.accumulate_all_parents_data_by_name(disease):
            parents.append(denormalize(parent) + ("\n\n" + comment if comment.strip() else ""))

    if method == "deep":
        return [symptoms_join] + descriptions + parents

    else:
        raise Exception(f"Sth went terribly wrong")

In [13]:
query_for_diseases = "тревога, импульсивность, обман или воровство"

In [14]:
get_diseases_data_from_onto(query_for_diseases, onto)

['синдром зависимости',
 'синдром отмены',
 'употребление с вредными последствиями',
 'ипохондрическое расстройство',
 'неврастения',
 'расстройство сексуального созревания',
 'смешанное тревожное и депрессивное расстройство',
 'эмоционально неустойчивое расстройство личности']

In [15]:
get_diseases_data_from_onto(query_for_diseases, onto, method="shallow_described")

['тревога,  импульсивность,  обман или воровство',
 'Группа поведенческих, мнестических и физиологических феноменов, развивающихся при неоднократном использования вещества, которые включают сильное желание принять наркотик, отсутствие самоконтроля, употребление на смотря на пагубные последствия, более высокий приоритет употребления наркотиков перед другими действиями и обязательствами, увеличенную толерантность к веществам.',
 'Группа симптомов различного сочетания и степени тяжести, возникающих при абсолютной или относительной отмены употребления психоактивного вещества после постоянного применения этого вещества.',
 'Употребление психотропного вещества, которое наносит ущерб здоровью. Повреждение может быть физическим (как в случаях гепатита от самоназначения введенных психотропных веществ) или психическим (например, эпизоды депрессивного расстройства при длительном употреблении алкоголя).',
 'Важнейшей чертой является устойчивая озабоченность пациента возможностью иметь у себя тяжел

In [16]:
get_diseases_data_from_onto(query_for_diseases, onto, method="deep")

['тревога,  импульсивность,  обман или воровство',
 'Группа поведенческих, мнестических и физиологических феноменов, развивающихся при неоднократном использования вещества, которые включают сильное желание принять наркотик, отсутствие самоконтроля, употребление на смотря на пагубные последствия, более высокий приоритет употребления наркотиков перед другими действиями и обязательствами, увеличенную толерантность к веществам.',
 'Группа симптомов различного сочетания и степени тяжести, возникающих при абсолютной или относительной отмены употребления психоактивного вещества после постоянного применения этого вещества.',
 'Употребление психотропного вещества, которое наносит ущерб здоровью. Повреждение может быть физическим (как в случаях гепатита от самоназначения введенных психотропных веществ) или психическим (например, эпизоды депрессивного расстройства при длительном употреблении алкоголя).',
 'Важнейшей чертой является устойчивая озабоченность пациента возможностью иметь у себя тяжел

In [17]:
def get_symptoms_data_from_onto(
    query: str,
    onto: CogniScanOntoWrapper,
    delimiter = ",",
    method=None,  # ignored
):

    diseases = query.split(delimiter)

    symptoms = []
    for disease in diseases:
        if disease.strip():
            symptoms.extend(
                onto.get_symptoms_by_disease(
                    disease.strip()
                )
            )

    return list(map(denormalize, symptoms))

In [18]:
query_for_symptoms = "параноидная шизофрения, расстройства поведения"

In [19]:
get_symptoms_data_from_onto(query_for_symptoms, onto,)

['эмоциональная нестабильность',
 'бредовое восприятие',
 'паранойя',
 'слуховые галлюцинации']

### Ранжирование документов

In [20]:
def get_top_k_docs(
    query: str,
    for_:str,  # вместо этого можно потенциально вкрячить модель
    encoder: Encoder,
    index: RamIndex,
    onto: CogniScanOntoWrapper,
    delimiter = ",",
    method="shallow",
    k=3,
):

    if for_ == "симптомы":
        to_embed = get_diseases_data_from_onto(query, onto, delimiter=delimiter, method=method)
    elif for_ == "болезни":
        to_embed = get_symptoms_data_from_onto(query, onto, delimiter=delimiter, method=method)
    else:
        raise ValueError(f"Unknown argument: {for_}")

    return index.search(to_embed, encoder, k=k)


In [21]:
query_for_diseases = "тревога, импульсивность, обман или воровство"

In [22]:
get_top_k_docs(query_for_diseases, "симптомы", encoder, index, onto, k=5)

[{'score': np.float32(0.79962003),
  'text': 'Цель исследования — опыт реализации структурно-функционального подхода к хронической ишемии мозга (ХИМ) у пожилых лиц посредством комплексирования одновременно производимых измерений. Объект исследования — пожилые больные с ХИМ I стадии. В результате показан методологический опыт системного исследования ХИМ с позиций структурно-функционального подхода у пожилых лиц. Так, продемонстрировано, что прогрессирование хронических церебрально-дисгемических морфологических изменений в виде перивентрикулярного и субкортикального лейкоареоза (по данным МРТ) коррелирует со степенью выраженности синдрома когнитивной дисфункции (по данным Краткого ориентировочного теста). Для пациентов с ХИМ I стадии характерна гипергомоцистеинемия, что коррелирует с незначительно выраженной когнитивной дисфункцией. Выявленные биохимические, нейровизуализационные, нейропсихологические особенности у больных с ХИМ I стадии целесообразно учитывать при медико-психологической

In [23]:
get_top_k_docs(query_for_diseases, "симптомы", encoder, index, onto, k=5, method="shallow_described")

[{'score': np.float32(0.9605771),
  'text': 'Тревога и связанные с ней расстройства представляют наиболее распространенный тип психических нарушений как в общей популяции, так и в неврологической клинике. Современная систематика тревожных расстройств включает паническое расстройство, агорафобию, простые (специфические) фобии, социальное тревожное расстройство и генерализованное тревожное расстройство. Тревожные расстройства часто сопутствуют болезням нервной системы, ухудшают их течение и затрудняют лечение, выраженность тревоги обычно соответствует тяжести неврологических симптомов. Тревога нередко предшествует болезням мозга, но ответ на вопросы о том, способствует ли она этим болезням, является их предиктором или ранним проявлением, требует дополнительных исследований. Современные подходы к лечению тревоги предполагают применение бензодиазепинов, антидепрессантов, отдельных нормотимиков, антипсихотиков, а также использование психотерапии.',
  'meta': None},
 {'score': np.float32(0.9

In [24]:
get_top_k_docs(query_for_diseases, "симптомы", encoder, index, onto, k=5, method="deep")

[{'score': np.float32(0.9417591),
  'text': 'Тревожные расстройства являются наиболее распространенным типом психических нарушений. Они включают генерализованное тревожное расстройство, паническое расстройство, агорафобию, специфические фобии и социальное тревожное расстройство. Тревога и связанные с ней расстройства играют важную роль в формировании глобального бремени болезней, а распознавание и лечение этих расстройств имеют большое социальное и экономическое значение. Современные подходы к лечению тревожных состояний предполагают применение когнитивно-поведенческой терапии, антидепрессантов, бензодиазепинов, антипсихотиков, отдельных антиконвульсантов и некоторых других лекарственных средств.',
  'meta': None},
 {'score': np.float32(0.93800426),
  'text': 'Тревога и связанные с ней расстройства представляют наиболее распространенный тип психических нарушений как в общей популяции, так и в неврологической клинике. Современная систематика тревожных расстройств включает паническое рас

In [25]:
query_for_symptoms = "параноидная шизофрения, расстройства поведения"

In [26]:
get_top_k_docs(query_for_symptoms, "болезни", encoder, index, onto, k=5)

[{'score': np.float32(0.81795603),
  'text': 'В поисковых базах РИНЦ и PubMed запрошены публикации за последние 40 лет по запросам «флувоксамин», «тревожно-депрессивные расстройства», «тревога», «депрессия», «коморбидность», посвященные эффективности флувоксамина при различных вариантах расстройств тревожно-депрессивного спектра, тревожных депрессиях. Данные приведенных исследований свидетельствуют о том, что флувоксамин (Зоварт Сан) в дозах 50—300 мг/сут является высоко эффективным средством для лечения не только тревожных депрессий и генеза (психогенные, органические, смешанные, аутохтонно-эндогенные) и степени тяжести (вплоть до психотических), но и более широкого спектра тревожно-депрессивных расстройств, включающих расстройства адаптации, обсессивно-компульсивное расстройство, соматизированные, дисморфофобические, инсомнические симптомокомплексы и нарушения пищевого поведения. Широкий спектр клинического действия флувоксамина обусловлен его основным и дополнительными механизмами д